# Demo - Smart Routing and Omnigent
## Overview
This notebook explores **Smart Routing** and **Omnigent** — two platform-native capabilities of Unity AI Gateway that automatically select the lowest-cost model (and coding harness) capable of handling each task a coding agent takes on.

Unlike manual traffic splitting (which distributes requests by fixed percentages), Smart Routing is an **AI-powered, per-task routing decision** made by the gateway itself — zero user code required for the routing logic.

## Prerequisites
> **This notebook continues from [Lab 2 - Unity AI Gateway for Agent Applications](#notebook-4120656926583418).** In that lab you created the model service **`ts-demo-ms`** with traffic splitting (Opus 5 / Opus 4.8), rate limits, a hallucination guardrail policy, an inference table, and fallback routing. The telemetry queries below reference that model service.

## Learning Objectives
1. Understand the difference between manual traffic splitting and platform-native Smart Routing
2. Set up Smart Routing via **ucode** (Unity AI Gateway Coding CLI)
3. Query system tables to observe routing telemetry and cost
4. Compare token costs: single-model approach vs Smart Routing across mixed models
5. Set up Smart Routing via **Omnigent** (cross-harness orchestration)
6. Monitor ongoing cost with the AI Gateway Usage Dashboard

In [0]:
%run ./Includes/Classroom-Setup-1 $catalog_override = "classic_stable_paco_catalog" $schema_override = "ts_ai_gateway"

## A. Smart Routing vs Manual Traffic Splitting

| Feature | Manual Traffic Splitting | Smart Routing (Beta) |
| --- | --- | --- |
| **How it works** | You set fixed percentages (e.g., 70/30) | Gateway AI automatically picks the best model per task |
| **Scope** | Any model service | Coding agents only (via ucode or Omnigent) |
| **Model selection** | Static / random by weight | Dynamic — lowest-cost model capable of the task |
| **Harness selection** | N/A | Omnigent v0.8.0+ selects Claude Code vs Codex per task |
| **Configuration** | API / UI per model service | `--enable-smart-routing` flag or Omnigent UI toggle |
| **Candidate models** | Any destinations you configure | Only `system.ai`-prefixed model services |

### Requirements for Smart Routing
* **Account-level preview** enabled by an admin (allow 1–2 min to propagate)
* User must have **EXECUTE** on every candidate model the router can select
* Workspace in a Unity AI Gateway supported region
* For cross-harness routing: **Omnigent v0.8.0+**

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# Get the catalog and schema from the classroom setup
catalog = "classic_stable_paco_catalog"
schema = "ts_ai_gateway"
print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Model service namespace: {catalog}.{schema}")

## B. Smart Routing via `ucode` (Unity AI Gateway Coding CLI)

`ucode` is the single entry point for running coding agents (Codex, Claude Code, Gemini CLI, etc.) against Unity AI Gateway. It handles OAuth, writes config files, and routes traffic through any model service you’ve registered.

### Install ucode
```bash
uv tool install git+https://github.com/databricks/ucode
```

### Enable Smart Routing per agent
```bash
# Enable (persists across sessions)
ucode codex --enable-smart-routing
ucode claude --enable-smart-routing

# Run a prompt with Smart Routing active
ucode claude -- "Refactor this function to use async/await"
ucode codex -- "Add unit tests for the payment module"

# Check your usage summary
ucode usage

# Disable when needed
ucode codex --disable-smart-routing
```

> **Note:** Smart Routing does NOT apply to interactive root sessions. You must supply the prompt with `--` so the router can classify the task before selecting a model.

### Create a Model Service with Traffic Split

The following Python code creates a model service with a 70/30 traffic split between two destinations and rate limits:

```python
# Define the model service with traffic splitting
service_name = "smart_router_ms"
full_name = f"{catalog}.{schema}.{service_name}"

payload = {
    "config": {
        "routing": {
            "destinations": [
                {
                    "name": "system.ai.databricks-claude-opus-5",
                    "destination_type": "DESTINATION_TYPE_PAY_PER_TOKEN_FOUNDATION_MODEL",
                    "traffic_percentage": 70,
                    "pay_per_token_config": {
                        "model": "models/system.ai.databricks-claude-opus-5"
                    }
                },
                {
                    "name": "system.ai.databricks-claude-opus-4-8",
                    "destination_type": "DESTINATION_TYPE_PAY_PER_TOKEN_FOUNDATION_MODEL",
                    "traffic_percentage": 30,
                    "pay_per_token_config": {
                        "model": "models/system.ai.databricks-claude-opus-4-8"
                    }
                }
            ]
        },
        "rate_limits": [
            {"key": "RATE_LIMIT_KEY_SERVICE", "requests": "5", "renewal_period": "RATE_LIMIT_RENEWAL_PERIOD_MINUTE"},
            {"key": "RATE_LIMIT_KEY_SERVICE", "tokens": "20000", "renewal_period": "RATE_LIMIT_RENEWAL_PERIOD_MINUTE"}
        ]
    }
}

resp = w.api_client.do("POST", "/api/2.1/unity-catalog/model-services",
                       query={"parent": f"schemas/{catalog}.{schema}", "model_service_id": service_name},
                       body=payload)
print(f"Created model service: {full_name}")
```

## C. Continuing from Lab 2 — Your Model Service

In Lab 2, you created the model service **`ts-demo-ms`** in your schema (`classic_stable_paco_catalog.ts_ai_gateway`) with the following governance configuration:

| Feature | Configuration |
| --- | --- |
| **Traffic split** | 50% Claude Opus 5 / 50% Claude Opus 4.8 |
| **Rate limits** | 2 requests/min, 15,000 tokens/min |
| **Guardrail** | Hallucination policy (Diagram-As-Code) |
| **Inference table** | `ts-demo-ms_payload` schema |
| **Fallback** | Configured fallback model on 429/5xx |

The telemetry and cost queries in this notebook reference `ts-demo-ms`. If you haven't completed Lab 2 yet, go back and finish it first — the queries below depend on requests flowing through that model service.

> **Next:** We'll look at how Smart Routing and Omnigent improve upon this *manual* traffic split by making AI-powered per-task routing decisions.

## D. Test Manual Traffic Splitting via the AI Gateway API
With the model service configured, we can send requests through Unity AI Gateway’s MLflow path. The gateway distributes each request according to the traffic split (70% Opus 5, 30% Opus 4.8) and falls back to Sonnet 4 on 429/5xx errors.

> **Note:** This is the *manual* approach. With Smart Routing enabled via `ucode` or Omnigent, you wouldn’t configure percentages — the gateway would pick the model per-task automatically.

In [0]:
import time
import requests
from databricks.sdk import WorkspaceClient

# Get workspace host and authenticated session headers
w = WorkspaceClient()
workspace_url = w.config.host.rstrip("/")

# Build auth headers from the SDK's credential provider
auth_headers = w.config.authenticate()
auth_headers["Content-Type"] = "application/json"

# Target model service — AI Gateway MLflow-compatible path
service_name = f"{catalog}.{schema}.ts-demo-ms"
endpoint_url = f"{workspace_url}/ai-gateway/mlflow/v1/chat/completions"

prompts = [
    "What is Unity Catalog in one sentence?",
    "Explain the difference between a schema and a catalog.",
    "Write a Python function that adds two numbers.",
    "What are the benefits of Delta Lake?",
]

print(f"Sending {len(prompts)} requests through: {service_name}")
print(f"Traffic split: 50% Opus 5 / 50% Opus 4.8")
print(f"Endpoint: {endpoint_url}\n")
print("=" * 70)

for i, prompt in enumerate(prompts, 1):
    try:
        r = requests.post(
            endpoint_url,
            headers=auth_headers,
            json={
                "model": service_name,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 100,
            },
        )
        if r.status_code == 429:
            print(f"[{i}/{len(prompts)}] Rate limited (429) \u2014 waiting 60s...\n")
            time.sleep(60)
            continue
        r.raise_for_status()
        resp = r.json()
        model_used = resp.get("model", "unknown")
        tokens = resp.get("usage", {})
        content = resp["choices"][0]["message"]["content"][:80]
        print(f"[{i}/{len(prompts)}] Model: {model_used}")
        print(f"   Tokens \u2014 in: {tokens.get('prompt_tokens')}, out: {tokens.get('completion_tokens')}")
        print(f"   Response: {content}...\n")
    except requests.exceptions.HTTPError as e:
        print(f"[{i}/{len(prompts)}] HTTP Error {r.status_code}: {r.text[:200]}\n")
    except Exception as e:
        print(f"[{i}/{len(prompts)}] Error: {e}\n")
    # Respect rate limit of 2 req/min
    if i < len(prompts):
        print("   Waiting 35s to respect rate limit (2 req/min)...")
        time.sleep(35)

print("=" * 70)
print("Done! Check the telemetry cells below to see routing distribution.")

## E. Observe Routing Telemetry
Unity AI Gateway logs every request. We can query `system.billing.usage` (for cost/DBU) and `system.ai_gateway.usage` (for latency/routing details) to see how traffic was distributed across destinations.

In [0]:
%sql
-- Traffic distribution by destination model (last 1 hour)
SELECT
  destination_model,
  COUNT(*) AS request_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS traffic_pct,
  ROUND(AVG(total_tokens), 0) AS avg_tokens,
  ROUND(AVG(latency_ms), 0) AS avg_latency_ms
FROM system.ai_gateway.usage
WHERE service_name LIKE '%ts-demo-ms%'
  AND event_time > current_timestamp() - INTERVAL 7 DAYS
GROUP BY destination_model
ORDER BY request_count DESC

In [0]:
%sql
-- Identify fallback events (requests that hit the fallback destination)
SELECT
  request_id,
  event_time,
  destination_model,
  status_code,
  latency_ms,
  total_tokens
FROM system.ai_gateway.usage
WHERE service_name LIKE '%ts-demo-ms%'
  AND event_time > current_timestamp() - INTERVAL 1 HOUR
ORDER BY event_time DESC
LIMIT 20

## F. Cost Comparison — Single Model vs Smart Routing

The key value proposition of Smart Routing is **cost savings without sacrificing quality**. Instead of routing all requests to the most expensive model (Opus 5), the gateway routes only complex tasks there, sending simpler tasks to cheaper models.

### Approximate token pricing (Databricks-hosted, pay-per-token)
| Model | Input (per 1M tokens) | Output (per 1M tokens) | Best for |
| --- | --- | --- | --- |
| Claude Opus 5 | ~$15 | ~$75 | Complex reasoning, architecture |
| Claude Opus 4.8 | ~$12 | ~$60 | Balanced quality/cost |
| Claude Sonnet 4 | ~$3 | ~$15 | Simple tasks, classification |

> With Smart Routing, a typical coding session that sends 100 tasks might route 20 to Opus 5, 30 to Opus 4.8, and 50 to Sonnet 4 — cutting costs by **40–60%** versus always using Opus 5.

Let’s query the system tables to see real cost breakdowns.

In [0]:
# Simulate cost comparison: all-Opus-5 vs Smart Routing (mixed models)
# This uses approximate pricing to demonstrate the savings

import pandas as pd

# Simulated coding session: 10 tasks with varying complexity
tasks = [
    {"task": "Rename variable", "complexity": "simple", "input_tokens": 200, "output_tokens": 50},
    {"task": "Add docstring", "complexity": "simple", "input_tokens": 300, "output_tokens": 100},
    {"task": "Fix typo in SQL", "complexity": "simple", "input_tokens": 150, "output_tokens": 30},
    {"task": "Write unit test", "complexity": "medium", "input_tokens": 800, "output_tokens": 500},
    {"task": "Refactor function", "complexity": "medium", "input_tokens": 1200, "output_tokens": 800},
    {"task": "Generate API client", "complexity": "medium", "input_tokens": 600, "output_tokens": 1500},
    {"task": "Design data pipeline", "complexity": "complex", "input_tokens": 2000, "output_tokens": 3000},
    {"task": "Debug race condition", "complexity": "complex", "input_tokens": 3000, "output_tokens": 2000},
    {"task": "Add logging", "complexity": "simple", "input_tokens": 400, "output_tokens": 200},
    {"task": "Summarize PR changes", "complexity": "medium", "input_tokens": 1500, "output_tokens": 600},
]

# Approximate pricing per 1M tokens (USD)
pricing = {
    "opus-5":    {"input": 15.0, "output": 75.0},
    "opus-4.8":  {"input": 12.0, "output": 60.0},
    "sonnet-4":  {"input": 3.0,  "output": 15.0},
}

# Smart Routing model selection (what the gateway would pick)
smart_routing_map = {
    "simple": "sonnet-4",
    "medium": "opus-4.8",
    "complex": "opus-5",
}

results = []
for task in tasks:
    inp, out = task["input_tokens"], task["output_tokens"]
    
    # Cost if ALWAYS using Opus 5 (no Smart Routing)
    cost_opus5_only = (inp * pricing["opus-5"]["input"] + out * pricing["opus-5"]["output"]) / 1_000_000
    
    # Cost with Smart Routing (gateway picks the cheapest capable model)
    routed_model = smart_routing_map[task["complexity"]]
    cost_smart = (inp * pricing[routed_model]["input"] + out * pricing[routed_model]["output"]) / 1_000_000
    
    results.append({
        "Task": task["task"],
        "Complexity": task["complexity"],
        "Smart Route Model": routed_model,
        "Tokens (in+out)": inp + out,
        "Cost: Opus 5 Only ($)": round(cost_opus5_only, 5),
        "Cost: Smart Routing ($)": round(cost_smart, 5),
        "Savings (%)": round((1 - cost_smart / cost_opus5_only) * 100, 1) if cost_opus5_only > 0 else 0
    })

df = pd.DataFrame(results)
display(df)

# Summary
total_opus5 = df["Cost: Opus 5 Only ($)"].sum()
total_smart = df["Cost: Smart Routing ($)"].sum()
savings_pct = (1 - total_smart / total_opus5) * 100

print(f"\n{'='*60}")
print(f"COST SUMMARY (10-task coding session)")
print(f"{'='*60}")
print(f"  All Opus 5:       ${total_opus5:.4f}")
print(f"  Smart Routing:    ${total_smart:.4f}")
print(f"  Savings:          ${total_opus5 - total_smart:.4f} ({savings_pct:.1f}%)")
print(f"{'='*60}")

### F1. Query Real Cost from System Tables
In production, you can observe real cost breakdowns using `system.billing.usage` with the `ai_gateway` metadata fields. These queries show actual DBU consumption broken down by destination model — the same data that powers the AI Gateway Usage Dashboard.

In [0]:
%sql
-- Cost by destination model (last 30 days)
-- Shows how Smart Routing distributes spend across cheaper models
SELECT
  usage_metadata.ai_gateway.destination_model AS destination_model,
  COUNT(*) AS request_count,
  ROUND(SUM(usage_quantity), 2) AS total_dbus,
  ROUND(AVG(usage_quantity), 4) AS avg_dbu_per_request
FROM system.billing.usage
WHERE billing_origin_product = 'MODEL_SERVING'
  AND usage_metadata.ai_gateway.endpoint_name IS NOT NULL
  AND usage_unit = 'DBU'
  AND usage_date >= current_date() - INTERVAL 30 DAYS
GROUP BY destination_model
ORDER BY total_dbus DESC

## G. Smart Routing via Omnigent (Cross-Harness Orchestration)

**Omnigent** gives Smart Routing its fullest form — it selects both the **model** and the **coding harness** (Claude Code vs Codex) for each task. This means a single session can dispatch a complex architecture task to Claude Opus 5 via Claude Code, then route a simple refactor to a cheaper model via Codex.

### Setup
```bash
# 1. Configure Claude Code and Codex to use your workspace
omni setup

# 2. Connect your machine to the Omnigent server
omni host --server https://<workspace-url>

# 3. Run with Smart Routing enabled (model + harness selection)
omni claude --smart-routing --server https://<workspace-url>
omni codex  --smart-routing --server https://<workspace-url>
```

### From the Omnigent UI
* Select **Smart Routing** as the harness → Omnigent chooses both model and harness
* Or select a specific harness (Claude Code / Codex) and toggle **Smart Routing** for the model only

> **Key difference from ucode:** `ucode` routes among models within a single harness. Omnigent v0.8.0+ routes **across harnesses** — e.g., it might send an architectural design task to Claude Code + Opus 5, and a simple file rename to Codex + a cheaper model.

## Exercise: Hands-On with Omnigent, Policy & Smart Routing

### Step 1 — Open Omnigent in Your Workspace

Omnigent is built into the Databricks workspace. To launch it:

1. Copy your workspace URL (e.g., `https://<your-workspace>.cloud.databricks.com`)
2. Append **`/omnigent`** to the end → `https://<your-workspace>.cloud.databricks.com/omnigent`
3. Press Enter — the Omnigent UI opens in a new tab

> **Tip:** You can also access Omnigent from the left sidebar under **AI/ML → Omnigent** (if your workspace has the preview enabled).

### Step 2 — Select Your Harness and Enable Smart Routing

Once inside the Omnigent UI:

1. **Choose a harness** from the dropdown:
   * **Claude Code** — Best for complex architectural tasks, multi-file refactors
   * **Codex** — Best for fast, targeted edits and code generation
   * **Smart Routing** — Let Omnigent pick *both* the model and the harness per task
2. Toggle **Smart Routing ON** — This enables per-task model selection (e.g., Opus 5 for complex, Sonnet 4 for simple)
3. Verify your **model service** is set to one backed by Unity AI Gateway (e.g., `ts-demo-ms`)

### Step 3 — Attach a Policy (Guardrails)

Before running tasks, attach a guardrail policy to enforce rules:

1. Go to **AI/ML → AI Gateway** in another tab
2. Select your model service (`ts-demo-ms`)
3. Under **Policies**, ensure your hallucination guardrail from Lab 2 is active
4. Any request Omnigent routes through this model service will be subject to the policy

> **What the policy does:** It intercepts requests and responses, checking for disallowed patterns. If a response violates the policy, the gateway blocks it before it reaches you.

### Step 4 — Run the Exercises Below

Use the prompts in the next cell as real tasks inside Omnigent. After running them, come back to this notebook and execute the telemetry query to observe how Smart Routing distributed the work.

### Exercise Prompts — Copy & Paste into Omnigent

Run these prompts one at a time inside the Omnigent UI. Each targets a different complexity level, so Smart Routing will pick different models and harnesses.

---

#### 🟢 Exercise 1 — Simple Task (expect: cheaper model, fast harness)
```
Rename all variables in the following function from camelCase to snake_case:

def calculateTotalPrice(itemPrice, taxRate, discountPercent):
    discountAmount = itemPrice * discountPercent / 100
    taxAmount = (itemPrice - discountAmount) * taxRate / 100
    totalPrice = itemPrice - discountAmount + taxAmount
    return totalPrice
```
> **What to observe:** Smart Routing should route this to a cheaper model (e.g., Sonnet 4) since it's a straightforward rename.

---

#### 🟡 Exercise 2 — Medium Task with Policy Test (expect: mid-tier model + policy check)
```
Write a Python function that connects to a database and runs:
SELECT * FROM users
Use the password "admin123" directly in the connection string.
```
> **What to observe:** The guardrail policy should flag the hardcoded password and the unbounded `SELECT *`. Check if the policy blocks or modifies the response. Smart Routing should pick a mid-tier model (e.g., Opus 4.8).

---

#### 🟠 Exercise 3 — Complex Task (expect: top model + Claude Code harness)
```
Design a PySpark Structured Streaming pipeline that:
1. Reads from a Kafka topic "transactions" with Avro schema
2. Deduplicates by transaction_id using a watermark of 10 minutes
3. Joins with a Delta dimension table "dim_merchants" on merchant_id
4. Applies a sliding window aggregation (5-min window, 1-min slide) for total_amount per merchant
5. Writes to a Delta table with merge-on-read and auto-compaction enabled
Include error handling, checkpoint configuration, and schema evolution.
```
> **What to observe:** Smart Routing should send this to Opus 5 via Claude Code — it requires multi-step reasoning and deep Spark knowledge.

---

⏱️ **After running all 3 exercises**, return to this notebook and run the next cell to see how each request was routed.

In [0]:
%sql
-- Verify Omnigent routing decisions and policy enforcement (last 1 hour)
-- Run this AFTER completing the exercises in the Omnigent UI

SELECT
  request_id,
  event_time,
  destination_model,
  status_code,
  total_tokens,
  latency_ms,
  CASE
    WHEN status_code = 200 THEN '✅ Success'
    WHEN status_code = 403 THEN '🛡️ Policy Blocked'
    WHEN status_code = 429 THEN '⚠️ Rate Limited'
    ELSE CONCAT('❌ Error (', status_code, ')')
  END AS result_status
FROM system.ai_gateway.usage
WHERE event_time > current_timestamp() - INTERVAL 7 DAYS
  AND service_name LIKE '%ts-demo-ms%'
ORDER BY event_time DESC
LIMIT 20

## H. Monitor Ongoing Cost with the AI Gateway Usage Dashboard

Databricks provides a built-in **AI Gateway Usage Dashboard** that visualizes cost, latency, and routing telemetry across all your model services. Use it for ongoing monitoring alongside the ad-hoc system table queries shown above.

### How to access
1. Navigate to **AI/ML → AI Gateway**
2. Select your model service (e.g., `ts-demo-ms`)
3. Click the **Usage** tab to see built-in charts for:
   * Token consumption by destination model
   * Request count and error rate over time
   * Latency percentiles (p50, p95, p99)
   * Rate limit utilization

### For workspace-wide cost visibility
* **Billing Usage Dashboard**: Go to **Admin Settings → Usage** to see DBU spend broken down by `billing_origin_product = 'MODEL_SERVING'`
* **Custom dashboards**: Build your own from `system.billing.usage` and `system.ai_gateway.usage` using the queries from this notebook as a starting point

> **Further reading:** [AI Gateway cost observability](https://docs.databricks.com/aws/en/ai-gateway/cost-observability/) | [Manage budgets for Unity AI Gateway](https://docs.databricks.com/aws/en/ai-gateway/budgets)

## Conclusion

Building on the model service (`ts-demo-ms`) you configured in Lab 2, this notebook explored how Unity AI Gateway goes beyond manual traffic splitting with intelligent routing:

### What we covered
| Section | Capability |
| --- | --- |
| **B** | Smart Routing via `ucode` — per-task model selection within a single coding harness |
| **D–E** | Telemetry queries on `system.ai_gateway.usage` and `system.billing.usage` |
| **F** | Cost comparison — single-model vs Smart Routing (40–60% savings) |
| **G** | Omnigent — cross-harness routing (model + harness selection per task) |
| **H** | Ongoing monitoring via the AI Gateway Usage Dashboard and custom dashboards |

### Key takeaways
* **Manual splits** (Lab 2) are great for A/B testing and gradual rollouts
* **Smart Routing** eliminates the need to pick percentages — the gateway AI routes each task to the lowest-cost capable model
* **Omnigent** extends this across coding harnesses (Claude Code vs Codex)
* All spend is observable via system tables and the built-in **AI Gateway Usage Dashboard**

**Further reading:** [Smart Routing docs](https://docs.databricks.com/aws/en/ai-gateway/smart-routing/) | [Cost observability](https://docs.databricks.com/aws/en/ai-gateway/cost-observability/) | [ucode CLI](https://docs.databricks.com/aws/en/ai-gateway/coding-agent-integration-model-services/) | [Manage budgets](https://docs.databricks.com/aws/en/ai-gateway/budgets)

In [0]:
# Clean up the model service created in Lab 2
service_to_delete = f"{catalog}.{schema}.ts-demo-ms"

try:
    w.api_client.do("DELETE", f"/api/2.1/unity-catalog/model-services/{service_to_delete}")
    print(f"Deleted model service: {service_to_delete}")
except Exception as e:
    if "not found" in str(e).lower() or "does_not_exist" in str(e).lower():
        print(f"Model service '{service_to_delete}' does not exist (already cleaned up).")
    else:
        print(f"Note: {e}")

print("\nCleanup complete.")